# Study 18 — The Structure of Snooker

## From rules to a game tree

Previous studies have often treated a snooker frame as a single event: one player wins and the other loses.

That is a useful abstraction, but a frame is itself the result of a sequence of smaller events governed by the rules of snooker.

Before building more detailed models, this study steps back and asks:

> **What is the structure of a game of snooker?**

We will describe how a match, frame, visit and shot fit together, and use that structure to begin thinking about snooker as a game tree.

The aim is not yet to model player ability or predict outcomes. It is to understand the game that those later models are simplifying.

## The levels of the game

Snooker can be described as a hierarchy of increasingly smaller events:

$$
\text{match}
\rightarrow
\text{frame}
\rightarrow
\text{visit}
\rightarrow
\text{shot}
$$

A **match** is decided by the number of frames won.

A **frame** is one complete contest for points between the two players.

Within a frame, players alternate **visits** to the table.

A visit consists of one or more **shots** and continues while the player legally pots balls and retains the table.

This gives us a useful starting point for a game tree. The outcome of a match depends on the outcomes of its frames, while the outcome of each frame emerges from a sequence of visits and shots.

For now, we will work down through these levels rather than trying to represent the whole game at once.

## The structure of a frame

A standard frame begins with **15 reds** and **6 colours** on the table.

The balls have fixed point values:

| Ball | Points |
|---|---:|
| Red | 1 |
| Yellow | 2 |
| Green | 3 |
| Brown | 4 |
| Blue | 5 |
| Pink | 6 |
| Black | 7 |

While reds remain on the table, the normal scoring sequence alternates between:

$$
\text{red}
\rightarrow
\text{colour}
\rightarrow
\text{red}
\rightarrow
\text{colour}
\rightarrow \cdots
$$

A potted red stays off the table. A colour potted while reds remain is normally respotted.

After the final red has been potted, the player may still attempt a colour. The colours are then cleared in ascending value:

$$
\text{yellow}
\rightarrow
\text{green}
\rightarrow
\text{brown}
\rightarrow
\text{blue}
\rightarrow
\text{pink}
\rightarrow
\text{black}
$$

The frame therefore has two distinct scoring phases:

1. **Red–colour phase** — the number of reds remaining gradually falls.
2. **Colours phase** — the six colours are removed in a fixed order.

This shrinking set of balls is important for the game tree: the possible future states of a frame depend partly on how far through this sequence the players have progressed.

## A visit as a branching process

A player remains at the table while they continue to make legal scoring shots.

At a simplified level, each shot can lead to one of several outcomes:

$$
\text{shot}
\rightarrow
\begin{cases}
\text{legal pot} & \rightarrow \text{visit continues} \\
\text{miss or no pot} & \rightarrow \text{opponent's visit} \\
\text{foul} & \rightarrow \text{score/state changes and opponent normally plays}
\end{cases}
$$

A successful pot does not simply add points. It also changes what the player must attempt next.

For example, while reds remain:

$$
\text{pot red}
\rightarrow
\text{choose a colour}
\rightarrow
\text{pot colour}
\rightarrow
\text{attempt another red}
$$

A visit can therefore contain many shots before control of the table passes to the opponent.

This is the first important source of branching in the game tree. From any particular state, several different shot outcomes can produce different subsequent states.

## What defines the state of a frame?

To represent snooker as a game tree, we need to know what information is required to describe the current position.

At a simplified level, a frame state might include:

- the score of Player A;
- the score of Player B;
- the number of reds remaining;
- which colours remain on the table;
- which player is at the table;
- whether the next legal target is a red or a colour;
- and, during the final colours phase, which colour is next.

We can think of the state as:

$$
S =
(
\text{score}_A,
\text{score}_B,
\text{reds remaining},
\text{colours remaining},
\text{player to act},
\text{ball on}
)
$$

A shot then moves the frame from one state to another:

$$
S_t
\rightarrow
S_{t+1}
$$

Different shot outcomes can lead to different next states, so repeated transitions generate a tree of possible ways the frame can unfold.

This is already a substantial simplification. A real snooker position also depends on the physical locations of the balls, because those positions determine which shots are available and how difficult they are.

For now, however, we will separate the **rule state** of the frame from the full physical state of the table.

In [1]:
from dataclasses import dataclass
from typing import Tuple

# Represent a simplified snooker frame state.
#
# This deliberately ignores the physical positions of the balls.
# It only stores the information needed to describe the rule state
# of the frame at a particular moment.
@dataclass(frozen=True)
class SnookerState:
    score_a: int
    score_b: int
    reds_remaining: int
    colours_remaining: Tuple[str, ...]
    player_to_act: str
    ball_on: str


# Create the opening state of a standard frame:
# - both players have zero points;
# - all 15 reds remain;
# - all six colours are available;
# - Player A is at the table;
# - a red is the ball on.
initial_state = SnookerState(
    score_a=0,
    score_b=0,
    reds_remaining=15,
    colours_remaining=("yellow", "green", "brown", "blue", "pink", "black"),
    player_to_act="A",
    ball_on="red",
)

# Display the state so we can inspect the representation.
initial_state

SnookerState(score_a=0, score_b=0, reds_remaining=15, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='red')

In [2]:
# Define the point values of the colours.
COLOUR_VALUES = {
    "yellow": 2,
    "green": 3,
    "brown": 4,
    "blue": 5,
    "pink": 6,
    "black": 7,
}


def pot_red(state: SnookerState) -> SnookerState:
    """
    Return the next state after the player legally pots a red.

    This simplified transition:
    - adds one point to the player at the table;
    - removes one red;
    - keeps the same player at the table;
    - changes the ball on from red to colour.
    """

    # A red can only be potted when a red is on.
    if state.ball_on != "red":
        raise ValueError("A red is not currently the ball on.")

    # There must still be at least one red on the table.
    if state.reds_remaining <= 0:
        raise ValueError("No reds remain.")

    # Add the point to the player currently at the table.
    score_a = state.score_a
    score_b = state.score_b

    if state.player_to_act == "A":
        score_a += 1
    else:
        score_b += 1

    # Return the resulting rule state.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining - 1,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on="colour",
    )


# Apply the transition to the opening state.
after_red = pot_red(initial_state)

after_red

SnookerState(score_a=1, score_b=0, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='colour')

## Completing the red–colour cycle

Potting a red does not end the visit. It changes the ball on from a red to a colour.

The player may then nominate and attempt one of the six colours. If that colour is legally potted during the red–colour phase:

- its point value is added to the player's score;
- the colour is respotted;
- the same player remains at the table;
- and a red becomes the ball on again.

So a common scoring cycle is:

$$
\text{red}
\rightarrow
\text{colour}
\rightarrow
\text{red}
$$

For example, potting a red followed by the black scores:

$$
1 + 7 = 8
$$

and returns the frame to a state in which a red is on.

This gives us a second kind of state transition to represent in code.

In [3]:
def pot_colour(state: SnookerState, colour: str) -> SnookerState:
    """
    Return the next state after the player legally pots a colour.

    During the red-colour phase:
    - the colour scores its normal value;
    - the colour is respotted;
    - the same player remains at the table;
    - a red becomes the ball on again.

    After the final red has gone:
    - the colour is still respotted;
    - the colours phase then begins with yellow as the ball on.
    """

    # A colour can only be potted when a colour is on.
    if state.ball_on != "colour":
        raise ValueError("A colour is not currently the ball on.")

    # Check that the nominated colour is valid.
    if colour not in COLOUR_VALUES:
        raise ValueError(f"Unknown colour: {colour}")

    # Add the colour's value to the player currently at the table.
    score_a = state.score_a
    score_b = state.score_b

    if state.player_to_act == "A":
        score_a += COLOUR_VALUES[colour]
    else:
        score_b += COLOUR_VALUES[colour]

    # While reds remain, the colour is respotted
    # and a red becomes the ball on again.
    if state.reds_remaining > 0:
        next_ball_on = "red"

    # After the final red, the colour following it is respotted
    # and the ordered colours phase begins with yellow.
    else:
        next_ball_on = "yellow"

    # Return the resulting rule state.
    # colours_remaining does not change here because the colour
    # is respotted during this part of the frame.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on=next_ball_on,
    )


# Player A chooses and pots the black after the opening red.
after_black = pot_colour(after_red, "black")

# Inspect the new state.
after_black

SnookerState(score_a=8, score_b=0, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='red')

## When the visit ends

So far, we have followed only successful scoring shots.

Starting from the opening state, Player A has:

$$
0
\rightarrow
1
\rightarrow
8
$$

points after potting a red and then the black.

The corresponding rule states have changed from:

$$
\text{red on}
\rightarrow
\text{colour on}
\rightarrow
\text{red on}
$$

while Player A has remained at the table throughout.

But a visit does not continue indefinitely. If the player makes a legal shot without potting the ball on, the visit ends and the opponent comes to the table.

At our simplified level, this gives another transition:

$$
(\text{Player A to act})
\rightarrow
(\text{Player B to act})
$$

The scores and balls remaining may be unchanged, but the state is still different because control of the table has passed to the opponent.

This is important for the game tree. A shot can change the state even when no points are scored and no ball is removed.

In [4]:
def end_visit(state: SnookerState) -> SnookerState:
    """
    Return the next state after the player makes a legal
    non-scoring shot and their visit ends.

    In this simplified model:
    - no points are scored;
    - no balls are removed;
    - control passes to the opponent;
    - the ball on is determined by the phase of the frame.
    """

    # Switch the player at the table.
    next_player = "B" if state.player_to_act == "A" else "A"

    # While reds remain, every new turn begins with a red on.
    if state.reds_remaining > 0:
        next_ball_on = "red"

    # If the final red has gone and the colour following it
    # has just been played at, the ordered colours phase
    # begins with yellow.
    elif state.ball_on == "colour":
        next_ball_on = "yellow"

    # During the ordered colours phase, a legal miss leaves
    # the same colour on for the incoming player.
    else:
        next_ball_on = state.ball_on

    # Return the resulting rule state.
    return SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=next_player,
        ball_on=next_ball_on,
    )


# Player A fails to pot the next red.
after_miss = end_visit(after_black)

# Inspect the new state.
after_miss

SnookerState(score_a=8, score_b=0, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='red')

## The final red changes the structure

The red–colour cycle does not continue for the whole frame.

Once the **final red** has been potted, the player still plays a colour in the normal way. That colour is respotted, but this is the last time a colour is returned to the table as part of the red–colour sequence.

After that shot, the frame enters the ordered colours phase:

$$
\text{yellow}
\rightarrow
\text{green}
\rightarrow
\text{brown}
\rightarrow
\text{blue}
\rightarrow
\text{pink}
\rightarrow
\text{black}
$$

From this point, legally potted colours are **not respotted**. They are removed from the frame one by one.

This produces an important structural change in our state representation.

During the red–colour phase, all six colours remain available because they are respotted after being potted.

During the ordered colours phase, however, `colours_remaining` begins to shrink.

So the game tree has a clear boundary:

$$
\text{repeating red–colour phase}
\rightarrow
\text{ordered colours phase}
\rightarrow
\text{frame end}
$$

The next step is to represent that second phase explicitly.

In [5]:
# Fix the order in which the colours must be taken
# once all of the reds have left the table.
ORDERED_COLOURS = (
    "yellow",
    "green",
    "brown",
    "blue",
    "pink",
    "black",
)


def pot_ordered_colour(state: SnookerState) -> SnookerState:
    """
    Return the next state after legally potting a colour
    during the ordered colours phase.

    In this phase:
    - the colour scores its normal value;
    - the colour is removed rather than respotted;
    - the same player remains at the table;
    - the next remaining colour becomes the ball on.
    """

    # The ordered colours phase can only begin once
    # there are no reds remaining.
    if state.reds_remaining != 0:
        raise ValueError("The ordered colours phase has not begun.")

    # There must still be at least one colour on the table.
    if not state.colours_remaining:
        raise ValueError("No colours remain.")

    # The first colour remaining is the only colour on.
    expected_colour = state.colours_remaining[0]

    if state.ball_on != expected_colour:
        raise ValueError(
            f"{expected_colour} should be the ball on, not {state.ball_on}."
        )

    # Add the value of the colour to the player at the table.
    score_a = state.score_a
    score_b = state.score_b

    if state.player_to_act == "A":
        score_a += COLOUR_VALUES[expected_colour]
    else:
        score_b += COLOUR_VALUES[expected_colour]

    # A legally potted colour now stays off the table.
    colours_remaining = state.colours_remaining[1:]

    # If another colour remains, it becomes the ball on.
    # Otherwise the normal sequence of balls has been completed.
    next_ball_on = (
        colours_remaining[0]
        if colours_remaining
        else "frame end"
    )

    # Return the resulting rule state.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=colours_remaining,
        player_to_act=state.player_to_act,
        ball_on=next_ball_on,
    )


# Construct a simple state at the start of the ordered colours phase.
colours_start = SnookerState(
    score_a=40,
    score_b=32,
    reds_remaining=0,
    colours_remaining=ORDERED_COLOURS,
    player_to_act="A",
    ball_on="yellow",
)

# Player A pots the yellow.
after_yellow = pot_ordered_colour(colours_start)

# Inspect the new state.
after_yellow

SnookerState(score_a=42, score_b=32, reds_remaining=0, colours_remaining=('green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='green')

## The final black is not always the end

During the ordered colours phase, each legally potted colour is removed and the next colour becomes the ball on.

For the example above:

$$
40\text{–}32
\rightarrow
42\text{–}32
$$

and:

$$
\text{yellow}
\rightarrow
\text{green}
$$

This process normally continues until the black is resolved.

However, the final black introduces an important exception.

If the scores are level after the black has been dealt with, the frame does not end. The black is respotted and the players contest a **respotted black** to decide the frame.

So the end of the ordered colours sequence can branch:

$$
\text{final black}
\rightarrow
\begin{cases}
\text{scores unequal} & \rightarrow \text{frame ends} \\
\text{scores level} & \rightarrow \text{respotted black}
\end{cases}
$$

Our current `pot_ordered_colour()` function does not yet represent this distinction.

That is the next transition we need to add.

In [6]:
# Update the ordered-colour transition so a tied final black
# creates a new decision state rather than automatically
# leaving the same player at the table.
def pot_ordered_colour(state: SnookerState) -> SnookerState:
    """
    Return the next state after legally potting a colour
    during the ordered colours phase.

    In this phase:
    - the colour scores its normal value;
    - the colour is removed rather than respotted;
    - the same player remains at the table while colours remain.

    If the final black is potted:
    - unequal scores end the frame;
    - level scores create a respotted-black state;
    - the next player is not yet determined.
    """

    # The ordered colours phase can only begin once
    # there are no reds remaining.
    if state.reds_remaining != 0:
        raise ValueError("The ordered colours phase has not begun.")

    # There must still be at least one colour on the table.
    if not state.colours_remaining:
        raise ValueError("No colours remain.")

    # The first remaining colour is the ball on.
    expected_colour = state.colours_remaining[0]

    if state.ball_on != expected_colour:
        raise ValueError(
            f"{expected_colour} should be the ball on, not {state.ball_on}."
        )

    # Add the value of the colour to the player at the table.
    score_a = state.score_a
    score_b = state.score_b

    if state.player_to_act == "A":
        score_a += COLOUR_VALUES[expected_colour]
    else:
        score_b += COLOUR_VALUES[expected_colour]

    # Remove the legally potted colour.
    colours_remaining = state.colours_remaining[1:]

    # Normally the same player continues if another colour remains.
    next_player = state.player_to_act

    # If colours remain, the next one becomes the ball on.
    if colours_remaining:
        next_ball_on = colours_remaining[0]

    # If the final black makes the scores level,
    # respot the black and leave the next player unresolved.
    elif score_a == score_b:
        colours_remaining = ("black",)
        next_ball_on = "respotted black"
        next_player = "undecided"

    # Otherwise the frame is complete.
    else:
        next_ball_on = "frame end"

    # Return the resulting rule state.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=colours_remaining,
        player_to_act=next_player,
        ball_on=next_ball_on,
    )


# Construct a state where Player A trails by seven
# with only the final black remaining.
#
# Potting the black will therefore make the scores level
# and create a respotted-black state.
final_black_tie = SnookerState(
    score_a=50,
    score_b=57,
    reds_remaining=0,
    colours_remaining=("black",),
    player_to_act="A",
    ball_on="black",
)


# Pot the final black and inspect the resulting state.
after_final_black = pot_ordered_colour(final_black_tie)

after_final_black

SnookerState(score_a=57, score_b=57, reds_remaining=0, colours_remaining=('black',), player_to_act='undecided', ball_on='respotted black')

## Respotted black is a new decision state

A tied score after the final black does not simply continue the previous visit.

Instead, the black is respotted and the players draw lots for the choice of who plays next. The selected player then plays from in-hand.

So the transition is better represented as:

$$
\text{final black produces a tie}
\rightarrow
\text{respotted black}
\rightarrow
\text{player to act is determined}
$$

This means that the identity of the next player is not inherited from the previous state.

Our current representation therefore needs a small correction: when a respotted black is created, `player_to_act` should temporarily represent an unresolved choice rather than one of the two players.

This is another example of why defining the game as a sequence of states is useful. Some transitions are produced by shots, while others are produced by the rules themselves.

## Resolving a respotted black

Once the black has been respotted and the next player has been determined, the frame becomes much simpler.

Only the black remains as an object ball.

Under the official WPBSA rules, the first **pot or foul** ends the frame.  
[WPBSA Rules — Section 3, Rule 4: End of Frame, Game or Match (p. 20)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=24)

So the remaining game tree is very small:

$$
\text{respotted black}
\rightarrow
\begin{cases}
\text{legal pot} & \rightarrow \text{striker wins frame} \\
\text{foul} & \rightarrow \text{opponent wins frame} \\
\text{legal non-scoring shot} & \rightarrow \text{opponent plays}
\end{cases}
$$

Unlike the earlier stages of a frame, there is no further scoring sequence to work through.

This is therefore a useful example of how the size and structure of the game tree depend strongly on the current state of the frame. A position with 15 reds remaining can have a very long future; a respotted-black position has only one scoring ball left and a much narrower set of possible continuations.

### A real respotted-black foul

This is not merely a theoretical edge case.

In the 2025 Players Championship semi-final between Kyren Wilson and Neil Robertson, Wilson made a clearance to force a respotted black. On his subsequent safety attempt, the cue ball went in-off in the middle pocket.

Because the foul occurred on the respotted black, the frame ended immediately and was awarded to Robertson.

[World Snooker Tour — *Warrior Wilson Battles Back To Make Final*](https://www.wst.tv/news/2025/march/21/warrior-wilson-battles-back-to-make-final/)

It is a useful real example of the terminal branch in the game tree:

$$
\text{respotted black}
\rightarrow
\text{foul}
\rightarrow
\text{opponent wins frame}
$$

In [7]:
# Resolve who will play first on the respotted black.
#
# The rules use a draw of lots to determine who has the choice
# of playing next. Our game-state model does not need to represent
# the draw itself; it only needs the resulting player to act.
def set_respotted_black_player(
    state: SnookerState,
    player: str,
) -> SnookerState:
    """
    Set the player who will play first on a respotted black.

    This transition:
    - does not change either score;
    - leaves the black on the table;
    - resolves player_to_act from 'undecided' to A or B.
    """

    # This transition is only valid in an unresolved
    # respotted-black state.
    if state.ball_on != "respotted black":
        raise ValueError("This is not a respotted-black state.")

    if state.player_to_act != "undecided":
        raise ValueError("The player to act has already been determined.")

    # Only the two players in the frame can be selected.
    if player not in ("A", "B"):
        raise ValueError("Player must be 'A' or 'B'.")

    # Return the resolved state.
    return SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=player,
        ball_on=state.ball_on,
    )


# Suppose the draw and subsequent choice result in
# Player B playing first.
respotted_black_ready = set_respotted_black_player(
    after_final_black,
    "B",
)

# Inspect the resolved state.
respotted_black_ready

SnookerState(score_a=57, score_b=57, reds_remaining=0, colours_remaining=('black',), player_to_act='B', ball_on='respotted black')

## The terminal branches of a respotted black

Once the player to act has been determined, play resumes with only the black remaining.

From this state, a shot has three structurally different outcomes:

$$
\text{respotted black}
\rightarrow
\begin{cases}
\text{legal pot} & \rightarrow \text{striker wins frame} \\
\text{foul} & \rightarrow \text{opponent wins frame} \\
\text{legal non-scoring shot} & \rightarrow \text{opponent plays}
\end{cases}
$$

The first two outcomes are **terminal states**: no further shots are required to determine the frame winner.

The third is not terminal. The score remains level, the black remains on the table, and control passes to the opponent.

This gives us a useful distinction for the game tree:

- **continuing states**, from which further play is possible;
- **terminal states**, in which the winner of the frame is already determined.

The same distinction will become important when we eventually move from individual shots to the complete tree of a frame.

In [8]:
def play_respotted_black(
    state: SnookerState,
    outcome: str,
):
    """
    Resolve one shot played with a respotted black.

    Allowed outcomes:
    - 'pot': the striker legally pots the black and wins the frame;
    - 'foul': the striker fouls and the opponent wins the frame;
    - 'safe': a legal non-scoring shot is played and play continues.

    The function returns:
    - the resulting SnookerState;
    - the frame winner, or None if the frame continues.
    """

    # This function only applies once the respotted black
    # is in play and the player to act has been determined.
    if state.ball_on != "respotted black":
        raise ValueError("This is not a respotted-black state.")

    if state.player_to_act not in ("A", "B"):
        raise ValueError("The player to act has not been determined.")

    if outcome not in ("pot", "foul", "safe"):
        raise ValueError("Outcome must be 'pot', 'foul', or 'safe'.")

    # Copy the current scores so the appropriate branch
    # can update them.
    score_a = state.score_a
    score_b = state.score_b

    # A legal pot scores seven points for the striker
    # and ends the frame immediately.
    if outcome == "pot":
        winner = state.player_to_act

        if winner == "A":
            score_a += 7
        else:
            score_b += 7

        next_state = SnookerState(
            score_a=score_a,
            score_b=score_b,
            reds_remaining=0,
            colours_remaining=(),
            player_to_act=state.player_to_act,
            ball_on="frame end",
        )

    # A foul gives seven penalty points to the opponent
    # and also ends the frame immediately.
    elif outcome == "foul":
        winner = "B" if state.player_to_act == "A" else "A"

        if winner == "A":
            score_a += 7
        else:
            score_b += 7

        next_state = SnookerState(
            score_a=score_a,
            score_b=score_b,
            reds_remaining=0,
            colours_remaining=(),
            player_to_act=state.player_to_act,
            ball_on="frame end",
        )

    # A legal non-scoring shot leaves the scores and black
    # unchanged but passes control to the opponent.
    else:
        winner = None
        next_player = "B" if state.player_to_act == "A" else "A"

        next_state = SnookerState(
            score_a=state.score_a,
            score_b=state.score_b,
            reds_remaining=0,
            colours_remaining=("black",),
            player_to_act=next_player,
            ball_on="respotted black",
        )

    return next_state, winner


# Test the continuing branch first:
# Player B plays a legal safety without potting the black.
after_respotted_safety, winner = play_respotted_black(
    respotted_black_ready,
    "safe",
)

after_respotted_safety, winner

(SnookerState(score_a=57, score_b=57, reds_remaining=0, colours_remaining=('black',), player_to_act='A', ball_on='respotted black'),
 None)

## A repeating state on the respotted black

After Player B plays a legal non-scoring shot, the frame remains level at:

$$
57\text{–}57
$$

The black remains on the table, and Player A becomes the player to act.

So the transition is:

$$
(A\text{ or }B,\ \text{respotted black})
\rightarrow
(\text{other player},\ \text{respotted black})
$$

with no change to the score or balls remaining.

This means the respotted-black part of the game can contain a repeated sequence of structurally similar states:

$$
A
\rightarrow
B
\rightarrow
A
\rightarrow
B
\rightarrow
\cdots
$$

until one player either legally pots the black or commits a foul.

So although we have been describing the structure as a **game tree**, the state representation reveals an important distinction:

- the **history of shots** forms a branching tree;
- the **game states themselves** can recur.

Two different sequences of shots can therefore lead to the same simplified rule state.

The next step is to confirm the two terminal branches: a legal pot and a foul.

In [9]:
# Test the first terminal branch:
# Player B legally pots the respotted black.
after_respotted_pot, pot_winner = play_respotted_black(
    respotted_black_ready,
    "pot",
)

print("Legal pot:")
print(after_respotted_pot)
print("Winner:", pot_winner)


# Test the second terminal branch:
# Player B fouls while playing the respotted black.
after_respotted_foul, foul_winner = play_respotted_black(
    respotted_black_ready,
    "foul",
)

print("\nFoul:")
print(after_respotted_foul)
print("Winner:", foul_winner)

Legal pot:
SnookerState(score_a=57, score_b=64, reds_remaining=0, colours_remaining=(), player_to_act='B', ball_on='frame end')
Winner: B

Foul:
SnookerState(score_a=64, score_b=57, reds_remaining=0, colours_remaining=(), player_to_act='B', ball_on='frame end')
Winner: A


## Fouls add another kind of state transition

So far, most of our transitions have been driven by whether a player pots a ball or loses the table.

Fouls are different because they can change the score **without a legal scoring pot**.

At a simplified level:

$$
\text{foul by Player A}
\rightarrow
\text{penalty points to Player B}
$$

and vice versa.

The penalty depends on the balls involved and is normally at least four points.

A foul can therefore change several parts of the state at once:

- the opponent's score may increase;
- control of the table may change;
- the ball on for the next shot must be determined;
- and some fouls introduce additional choices under the rules.

This means that a foul is not simply another version of a missed pot.

In game-tree terms, it creates a distinct family of transitions.

For the moment, we will begin with a deliberately simplified foul:

> a player commits a foul, the opponent receives a specified number of penalty points, and the opponent becomes the player to act.

We can then add the more complicated consequences separately rather than trying to reproduce the entire foul section of the rules at once.

In [10]:
def commit_simple_foul(
    state: SnookerState,
    penalty: int,
) -> SnookerState:
    """
    Return the next state after a simplified foul.

    This first foul model assumes:
    - the opponent receives the specified penalty points;
    - penalty points are between 4 and 7;
    - the opponent elects to play next;
    - no free ball, miss call, replacement of balls, or
      requirement for the offender to play again is modelled.

    Those additional branches will be handled separately.
    """

    # Standard foul penalties in this simplified representation
    # range from four to seven points.
    if penalty not in (4, 5, 6, 7):
        raise ValueError("Penalty must be 4, 5, 6, or 7 points.")

    # Identify the non-offending player.
    next_player = "B" if state.player_to_act == "A" else "A"

    # Copy the scores before awarding the penalty.
    score_a = state.score_a
    score_b = state.score_b

    # Penalty points are awarded to the non-offending player.
    if next_player == "A":
        score_a += penalty
    else:
        score_b += penalty

    # Determine the ball on for the incoming player.
    #
    # While reds remain, a new turn normally begins with a red on.
    if state.reds_remaining > 0:
        next_ball_on = "red"

    # If all reds have gone and the foul occurred on the colour
    # following the final red, the ordered colours phase begins
    # with yellow.
    elif state.ball_on == "colour":
        next_ball_on = "yellow"

    # During the ordered colours phase, the same colour remains on
    # unless some more complicated rule consequence applies.
    else:
        next_ball_on = state.ball_on

    # Return the simplified resulting state.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=next_player,
        ball_on=next_ball_on,
    )


# Start from the position after Player A's red-black opening.
# Player A is on 8, Player B is on 0, and a red is on.
#
# Suppose Player A now commits a four-point foul.
after_simple_foul = commit_simple_foul(
    after_black,
    penalty=4,
)

# Inspect the resulting state.
after_simple_foul

SnookerState(score_a=8, score_b=4, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='red')

## A foul does not necessarily determine who plays next

Our first foul transition assumed that the non-offending player takes the next stroke.

That is only one possible continuation.

After a foul, the non-offender may instead require the offender to play again. The rules can also introduce further branches depending on the resulting position.

Two particularly important cases are:

- if the incoming player is snookered after the foul, the referee may call a **free ball**;
- if the referee calls **FOUL AND A MISS**, the non-offender may have additional options, including requiring the offender to play again from the position left or, in applicable circumstances, from the original position with the balls replaced.

So our simple transition:

$$
\text{foul by A}
\rightarrow
\text{B plays}
$$

is really only one branch of a larger structure:

$$
\text{foul by A}
\rightarrow
\begin{cases}
\text{B plays from position left} \\
\text{A is required to play again} \\
\text{free-ball situation} \\
\text{foul-and-a-miss options}
\end{cases}
$$

[WPBSA Rules — Section 3, Rule 12: Snookered After a Foul (p. 28)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=32)

[WPBSA Rules — Section 3, Rule 13: Play Again (p. 29)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=33)

[WPBSA Rules — Section 3, Rule 14: Foul and a Miss (pp. 29–31)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=33)

This is an important increase in the structure of the game tree. The result of a foul depends not only on what happened during the stroke, but also on the resulting table position, the referee's ruling, and a subsequent choice by the non-offending player.

In [11]:
def require_offender_to_play_again(
    state: SnookerState,
    offender: str,
) -> SnookerState:
    """
    Return a simplified state where the player who committed
    the foul is required to play again.

    This transition assumes:
    - the foul penalty has already been added to the opponent's score;
    - the balls remain in the position left after the foul;
    - no free ball or foul-and-a-miss replacement is involved;
    - the offender must take the next stroke.
    """

    # Only the two players in the frame are valid.
    if offender not in ("A", "B"):
        raise ValueError("Offender must be 'A' or 'B'.")

    # Determine the ball on for the player being required
    # to play again.
    #
    # While reds remain, the next turn begins with a red on.
    if state.reds_remaining > 0:
        next_ball_on = "red"

    # If all reds have gone and the previous state was the
    # colour following the final red, the ordered colours
    # phase begins with yellow.
    elif state.ball_on == "colour":
        next_ball_on = "yellow"

    # Otherwise, during the ordered colours phase,
    # the current colour remains the ball on.
    else:
        next_ball_on = state.ball_on

    # Return the state with the offender back at the table.
    return SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=offender,
        ball_on=next_ball_on,
    )


# We already modelled Player A committing a four-point foul,
# giving Player B four penalty points.
#
# Suppose Player B now requires Player A to play again.
after_play_again = require_offender_to_play_again(
    after_simple_foul,
    offender="A",
)

# Inspect the resulting state.
after_play_again

SnookerState(score_a=8, score_b=4, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='red')

## Requiring the offender to play again

The foul has changed the score from:

$$
8\text{–}0
\rightarrow
8\text{–}4
$$

but the non-offending player has chosen not to take the next stroke.

Instead, Player A — the player who committed the foul — is required to play again.

So the resulting state is:

- Player A: 8
- Player B: 4
- 14 reds remaining
- Player A to act
- red on

This demonstrates an important feature of the game tree.

A change of score does not necessarily imply a change of player:

$$
\text{A fouls}
\rightarrow
\text{B receives penalty points}
\rightarrow
\text{A may still play next}
$$

The identity of the next player can therefore depend on a **choice made after the shot has finished**, rather than purely on the outcome of the shot itself.

This is different from the simpler transitions we modelled earlier, where a missed pot automatically passed control to the opponent.

The next complication is a **free ball**, where a foul can temporarily change which balls may legally be nominated as the ball on.

## A free ball changes the legal possibilities

A free ball is called when, following a foul, the cue-ball is snookered on the ball on.

The important point is that the **ball on itself does not change**.

Instead, if the non-offending player elects to play, they may nominate another ball as a **free ball**. That nominated ball is treated as the ball on for the stroke and acquires its value.

For example, if a red is on and the player receives a free ball, they may nominate a colour:

$$
\text{red on}
+
\text{free ball}
\rightarrow
\text{nominated colour treated as red}
$$

If the nominated colour is potted, it scores one point and is respotted.

This reveals a limitation in our current state representation.

So far we have stored only:

$$
\text{ball on}
$$

but a free-ball position requires us to distinguish between:

- the actual **ball on**;
- whether a **free ball is available**;
- and, once selected, the **nominated free ball**.

Our state therefore needs to become slightly richer before we can represent this branch correctly.

[WPBSA Rules — Section 3, Rule 12: Snookered After a Foul (p. 28)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=32)

In [12]:
# Extend the frame state so it can represent free-ball situations.
#
# The two new fields have default values, so the simpler states
# we created earlier still have the same natural interpretation:
# no free ball is available unless the rules create one.
@dataclass(frozen=True)
class SnookerState:
    score_a: int
    score_b: int
    reds_remaining: int
    colours_remaining: Tuple[str, ...]
    player_to_act: str
    ball_on: str
    free_ball_available: bool = False
    nominated_free_ball: str | None = None


# Recreate the opening state using the richer representation.
#
# We do not need to specify the two free-ball fields here because
# their default values are False and None.
initial_state_v2 = SnookerState(
    score_a=0,
    score_b=0,
    reds_remaining=15,
    colours_remaining=("yellow", "green", "brown", "blue", "pink", "black"),
    player_to_act="A",
    ball_on="red",
)

# Inspect the expanded state.
initial_state_v2

SnookerState(score_a=0, score_b=0, reds_remaining=15, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='red', free_ball_available=False, nominated_free_ball=None)

## From a free-ball call to a nomination

The expanded state lets us distinguish an ordinary position from one in which the referee has called a free ball.

For a normal opening position:

$$
\text{free ball available} = \text{False}
$$

and:

$$
\text{nominated free ball} = \text{None}
$$

After an appropriate foul, however, the same underlying rule state might instead have:

$$
\text{ball on} = \text{red}
$$

while:

$$
\text{free ball available} = \text{True}
$$

The player may then nominate another ball.

For example:

$$
\text{red on}
\rightarrow
\text{free ball called}
\rightarrow
\text{black nominated}
$$

The black has not literally become a red. Rather, it is treated as the ball on for that stroke and therefore carries the value of the red.

This gives us two separate transitions to represent:

1. the referee's ruling creates a **free-ball state**;
2. the player chooses which available ball to **nominate**.

Keeping those transitions separate is useful because the first is determined by the rules and the resulting position, while the second is a choice made by the player.

In [13]:
# Create a free-ball state after the referee has made the ruling.
#
# This function does not attempt to decide whether a free ball
# should be awarded. That depends on the physical table position,
# which our simplified rule state does not represent.
def call_free_ball(state: SnookerState) -> SnookerState:
    """
    Mark the current position as a free-ball situation.

    This transition:
    - leaves the scores unchanged;
    - leaves the balls unchanged;
    - leaves the underlying ball on unchanged;
    - records that the player may nominate a free ball.
    """

    # A free ball cannot already have been nominated.
    if state.nominated_free_ball is not None:
        raise ValueError("A free ball has already been nominated.")

    # Return the same rule state with free-ball availability added.
    return SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on=state.ball_on,
        free_ball_available=True,
        nominated_free_ball=None,
    )


def nominate_free_ball(
    state: SnookerState,
    ball: str,
) -> SnookerState:
    """
    Record the ball nominated as the free ball.

    The underlying ball on remains unchanged.
    The nominated ball is stored separately because it is only
    treated as the ball on for this particular stroke.
    """

    # A nomination is only possible after a free ball has been called.
    if not state.free_ball_available:
        raise ValueError("No free ball is available.")

    # Only a colour currently available on the table can be used
    # in this simplified red-on example.
    if ball not in state.colours_remaining:
        raise ValueError(f"{ball} is not available to nominate.")

    # Store the player's nomination without changing the actual ball on.
    return SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on=state.ball_on,
        free_ball_available=True,
        nominated_free_ball=ball,
    )


# Construct a simple post-foul position:
# Player A leads 8-4, Player B is at the table,
# a red is on, and the referee calls a free ball.
free_ball_position = SnookerState(
    score_a=8,
    score_b=4,
    reds_remaining=14,
    colours_remaining=("yellow", "green", "brown", "blue", "pink", "black"),
    player_to_act="B",
    ball_on="red",
)

free_ball_called = call_free_ball(free_ball_position)

# Player B nominates the black as the free ball.
black_nominated = nominate_free_ball(
    free_ball_called,
    "black",
)

# Inspect the resulting state.
black_nominated

SnookerState(score_a=8, score_b=4, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='red', free_ball_available=True, nominated_free_ball='black')

## Potting the nominated free ball

Suppose Player B legally pots the black that has been nominated as a free ball while a red is on.

The black is treated as though it were the ball on for that stroke.

Because the actual ball on is a red:

$$
\text{value of nominated black} = 1
$$

So Player B scores **one point**, not seven.

The black is then respotted.

Crucially, no actual red has been removed from the table. The number of reds therefore remains:

$$
14
$$

After the free ball has been potted, the player has effectively completed the red part of a normal red–colour sequence, so a colour is on next.

The transition is therefore:

$$
(8\text{–}4,\ 14\text{ reds},\ \text{red on})
\rightarrow
(8\text{–}5,\ 14\text{ reds},\ \text{colour on})
$$

This is an unusual transition compared with an ordinary red pot:

- one point is scored;
- no red disappears;
- the nominated colour is respotted;
- the visit continues with a colour on.

[WPBSA Rules — Section 3, Rule 12: Snookered After a Foul (p. 28)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=32)

The free-ball rule therefore creates a scoring path that cannot occur in an ordinary red–colour cycle.

In [14]:
def pot_free_ball_as_red(state: SnookerState) -> SnookerState:
    """
    Return the next state after legally potting a nominated
    free ball when a red is the actual ball on.

    In this case:
    - the nominated ball scores one point;
    - no actual red is removed;
    - the nominated colour is respotted;
    - the same player remains at the table;
    - a colour becomes the ball on;
    - the free-ball nomination is cleared.

    This function deliberately handles only the red-on case.
    Other free-ball situations can be represented separately.
    """

    # A free ball must have been called and nominated.
    if not state.free_ball_available:
        raise ValueError("No free ball is available.")

    if state.nominated_free_ball is None:
        raise ValueError("No free ball has been nominated.")

    # This transition models only the case where a red is on.
    if state.ball_on != "red":
        raise ValueError("This function only applies when a red is on.")

    # The nominated ball must be one of the colours
    # currently available on the table.
    if state.nominated_free_ball not in state.colours_remaining:
        raise ValueError("The nominated free ball is not available.")

    # Copy the current scores.
    score_a = state.score_a
    score_b = state.score_b

    # Because a red is the actual ball on, the nominated
    # free ball carries the value of a red: one point.
    if state.player_to_act == "A":
        score_a += 1
    else:
        score_b += 1

    # No actual red has been potted, so reds_remaining
    # does not change. The nominated colour is respotted,
    # so colours_remaining also stays unchanged.
    #
    # The successful free ball has fulfilled the red part
    # of the sequence, so a colour is now on.
    return SnookerState(
        score_a=score_a,
        score_b=score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on="colour",
        free_ball_available=False,
        nominated_free_ball=None,
    )


# Player B legally pots the nominated black as a free ball.
after_free_ball = pot_free_ball_as_red(black_nominated)

# Inspect the resulting state.
after_free_ball

SnookerState(score_a=8, score_b=5, reds_remaining=14, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='colour', free_ball_available=False, nominated_free_ball=None)

## A free ball can create an extra scoring opportunity

After Player B pots the black as a free ball, the state becomes:

$$
8\text{–}5
$$

with **14 actual reds still remaining** and a colour on.

That matters because the free ball has effectively acted as an additional red without removing one of the reds from the table.

In the exceptional case where a free ball is available before any red has been potted, a player can therefore score:

$$
1 + 7
$$

from the free-ball sequence and still have all 15 reds available.

If every subsequent red is followed by a black, and the colours are then cleared, the maximum possible break becomes:

$$
(1 + 7) + 15(1 + 7) + 27 = 155
$$

rather than the familiar 147.

This follows directly from the state transition we have just represented: the nominated free ball can score as a red while leaving the actual number of reds unchanged.

[WPBSA Rules — Section 3, Rule 12: Snookered After a Foul (p. 28)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=32)

So even a rare rule such as the free ball can alter the set of scoring paths available through a frame.

### Competitive breaks above 147

A 155 remains the theoretical maximum, but breaks above 147 have occurred in professional tournament play.

There are currently two recorded examples:

- **Jamie Burnett — 148**, against Leo Fernandez in qualifying for the 2004 UK Championship.
- **Ronnie O'Sullivan — 153**, against Ryan Day at the 2026 World Open.

Both required a free ball, demonstrating that the additional scoring path created by the rule can matter in real competitive snooker.

[WPBSA — Breaks Over 147](https://www.wpbsa.com/about-us/history/147-breaks/)

## Foul and a miss can restore an earlier table position

A **FOUL AND A MISS** introduces a different kind of transition.

In applicable circumstances, the non-offending player may require the offender to play again with the balls replaced as closely as possible to their positions before the previous stroke.

This does **not** return the entire game to its earlier state.

The penalty points from the foul remain on the scoreboard:

$$
\text{score before stroke}
\rightarrow
\text{foul penalty added}
$$

while the physical table position can be restored:

$$
\text{table before stroke}
\rightarrow
\text{unsuccessful stroke}
\rightarrow
\text{table restored}
$$

So after replacement we can have:

$$
\text{same table position}
+
\text{different score}
$$

This is important for our definition of a state.

Our current `SnookerState` records the score and the rule state, but deliberately ignores the physical positions of the balls. It therefore cannot distinguish between:

- playing again from the position left after the foul; and
- replacing the balls and playing again from the original position.

A complete model would need to include the physical table configuration as part of the state.

This also shows that the game does not simply progress by permanently moving away from every previous position. Under the rules, a previous **table position** can be recreated while the overall state of the frame has still changed.

[WPBSA Rules — Section 3, Rule 14: Foul and a Miss (pp. 29–31)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=33)

## A frame can end with balls still on the table

Our examples so far have tended to move towards the natural completion of the scoring sequence:

$$
\text{reds}
\rightarrow
\text{colours}
\rightarrow
\text{final black}
\rightarrow
\text{frame end}
$$

But clearing the table is not required for a frame to end.

A player who requires **penalty points** may offer to concede the frame. If the opponent accepts the concession, the frame ends immediately even though balls may remain on the table.

An offered concession does not itself end the frame: if the opponent chooses to play on, the concession becomes null and void.

So a terminal branch can occur while the table still contains playable balls:

$$
\text{live frame}
\rightarrow
\text{concession offered}
\rightarrow
\begin{cases}
\text{accepted} & \rightarrow \text{frame end} \\
\text{play continues} & \rightarrow \text{live frame}
\end{cases}
$$

The rules also provide other ways for a frame to finish before another scoring stroke is played.

When only the black remains and aggregate points are not relevant:

- the striker may claim the frame if leading by more than seven points;
- the frame is awarded to the non-striker if the non-striker leads by more than seven points.

This means that:

$$
\text{terminal state}
\neq
\text{empty table}
$$

A terminal state simply means that the winner of the frame has been determined under the rules.

That distinction matters for a game-tree model. Some branches terminate because the scoring sequence has been exhausted, while others terminate because the rules make further play unnecessary.

[WPBSA Rules — Section 2, Rule 1: Frame (p. 10)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=14)

[WPBSA Rules — Section 4, Rule 2: Conceding (p. 37)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=41)

In [15]:
def maximum_points_remaining(state: SnookerState) -> int:
    """
    Calculate the maximum number of points still available
    from legal pots in our simplified rule state.

    Penalty points from future fouls are not included.
    """

    # A completed frame has no scoring points remaining.
    if state.ball_on == "frame end":
        return 0

    # While reds remain, each red can still produce a maximum
    # of eight points: one for the red and seven for a black.
    #
    # The final six colours are then worth 27 points.
    if state.reds_remaining > 0:
        points = (
            state.reds_remaining * 8
            + sum(COLOUR_VALUES[colour] for colour in state.colours_remaining)
        )

        # If a colour is currently on because a red has just
        # been potted, that colour can add up to another seven
        # points before the remaining red-colour cycles.
        if state.ball_on == "colour":
            points += 7

        # A free ball with a red on can create an additional
        # one-point "red", followed by a colour worth up to seven.
        if state.free_ball_available and state.ball_on == "red":
            points += 8

        return points

    # After the final red has been potted, but before its
    # following colour has been played, up to seven points
    # are available before the ordered colours begin.
    if state.ball_on == "colour":
        return (
            7
            + sum(COLOUR_VALUES[colour] for colour in state.colours_remaining)
        )

    # During the ordered colours phase, the points remaining
    # are simply the values of the colours still on the table.
    if state.ball_on in COLOUR_VALUES:
        points = sum(
            COLOUR_VALUES[colour]
            for colour in state.colours_remaining
        )

        # If a free ball is available and another colour can
        # be nominated, it can add the value of the ball on
        # without removing that actual ball.
        if (
            state.free_ball_available
            and len(state.colours_remaining) > 1
        ):
            points += COLOUR_VALUES[state.ball_on]

        return points

    # A respotted black has seven scoring points available.
    if state.ball_on == "respotted black":
        return 7

    raise ValueError(f"Unrecognised ball-on state: {state.ball_on}")


def requires_penalty_points(
    state: SnookerState,
    player: str,
) -> bool:
    """
    Return True if the player trails by more than the maximum
    number of points still available from legal pots.
    """

    if player not in ("A", "B"):
        raise ValueError("Player must be 'A' or 'B'.")

    # Calculate how far the selected player is behind.
    if player == "A":
        deficit = state.score_b - state.score_a
    else:
        deficit = state.score_a - state.score_b

    # Penalty points are required only when the deficit is
    # greater than the points still available from legal pots.
    return deficit > maximum_points_remaining(state)


def accept_concession(
    state: SnookerState,
    conceding_player: str,
):
    """
    Return a terminal state after a valid concession is accepted.

    Under the rules, a frame cannot be conceded unless
    at least one player requires penalty points.
    """

    # Only one of the two players can offer the concession.
    if conceding_player not in ("A", "B"):
        raise ValueError("Conceding player must be 'A' or 'B'.")

    # A concession is not permitted while neither player
    # requires penalty points.
    if not (
        requires_penalty_points(state, "A")
        or requires_penalty_points(state, "B")
    ):
        raise ValueError(
            "A concession is not permitted until a player "
            "requires penalty points."
        )

    # The opponent wins if the concession is accepted.
    winner = "B" if conceding_player == "A" else "A"

    # The frame ends without removing the balls still on the table.
    terminal_state = SnookerState(
        score_a=state.score_a,
        score_b=state.score_b,
        reds_remaining=state.reds_remaining,
        colours_remaining=state.colours_remaining,
        player_to_act=state.player_to_act,
        ball_on="frame end",
        free_ball_available=False,
        nominated_free_ball=None,
    )

    return terminal_state, winner


# Player A leads 74-30 with two reds remaining.
#
# There are at most:
#
#   2 * 8 + 27 = 43
#
# points still available from legal pots.
#
# Player B trails by 44 points, so Player B requires
# penalty points and a concession may legally be offered.
concession_position = SnookerState(
    score_a=74,
    score_b=30,
    reds_remaining=2,
    colours_remaining=("yellow", "green", "brown", "blue", "pink", "black"),
    player_to_act="B",
    ball_on="red",
)

print("Points remaining:", maximum_points_remaining(concession_position))
print(
    "Player B requires penalty points:",
    requires_penalty_points(concession_position, "B"),
)

# Player B offers the concession and Player A accepts.
after_concession, concession_winner = accept_concession(
    concession_position,
    conceding_player="B",
)

print(after_concession)
print("Winner:", concession_winner)

Points remaining: 43
Player B requires penalty points: True
SnookerState(score_a=74, score_b=30, reds_remaining=2, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='frame end', free_ball_available=False, nominated_free_ball=None)
Winner: A


## Concession as a rule-dependent terminal state

In this example, Player A leads:

$$
74\text{–}30
$$

with two reds remaining.

The maximum number of points still available from legal pots is:

$$
2(1 + 7) + 27 = 43
$$

but Player B trails by:

$$
74 - 30 = 44
$$

So Player B requires at least one penalty point to recover the deficit.

That makes a concession permissible under the rules.

When Player B's concession is accepted, the frame enters a terminal state even though:

- two reds remain;
- all six colours remain;
- and further legal scoring shots would still be possible.

So the terminal condition is created entirely by the rules rather than by exhausting the balls on the table.

Our model can now distinguish between:

$$
\text{frame mathematically recoverable}
$$

and:

$$
\text{penalty points required}
$$

which is precisely the boundary used by the concession rule.

It also gives us a useful validation check: if neither player requires penalty points, the model should reject an attempted concession rather than treating it as a legitimate terminal branch.

In [16]:
# Test a concession before either player requires penalty points.
#
# Player A leads 70-30 with two reds remaining.
# There are still:
#
#   2 * 8 + 27 = 43
#
# points available from legal pots.
#
# Player B trails by only 40, so neither player requires
# penalty points and a concession is not permitted.
early_concession_position = SnookerState(
    score_a=70,
    score_b=30,
    reds_remaining=2,
    colours_remaining=("yellow", "green", "brown", "blue", "pink", "black"),
    player_to_act="B",
    ball_on="red",
)

print(
    "Points remaining:",
    maximum_points_remaining(early_concession_position),
)
print(
    "Player B requires penalty points:",
    requires_penalty_points(early_concession_position, "B"),
)

# The model should reject this attempted concession.
try:
    accept_concession(
        early_concession_position,
        conceding_player="B",
    )
except ValueError as error:
    print("Concession rejected:", error)

Points remaining: 43
Player B requires penalty points: False
Concession rejected: A concession is not permitted until a player requires penalty points.


## Illegal actions are not branches of the legal game tree

The 70–30 example provides a useful contrast with the valid concession at 74–30.

With two reds remaining, there are still:

$$
43
$$

points available from legal pots.

At 70–30, Player B trails by only:

$$
70 - 30 = 40
$$

so the frame remains recoverable without penalty points.

Our model therefore rejects the attempted concession:

    Concession rejected: A concession is not permitted until a player requires penalty points.

This suggests another useful distinction when constructing the game tree.

Not every action that a player could physically attempt is a **legal branch** from the current state.

The rules constrain the set of permitted transitions:

$$
S_t
\rightarrow
\{\text{legal next states}\}
$$

An attempted early concession therefore does not create an ordinary terminal branch. It falls outside the legal transition set and is instead dealt with under the rules governing unsporting conduct.

[WPBSA Rules — Section 4, Rule 2: Conceding (p. 37)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=41)

This is an important part of representing any sport as a game tree: the rules do not merely determine how states change; they also determine which transitions are available at all.

## A stalemate can reset the frame

Snooker also contains a transition that is unlike the scoring, foul and concession branches considered so far.

If the referee considers that a **stalemate** exists, or is being approached, the players may be offered the option of restarting the frame. This is commonly called a **re-rack**.

If either player objects, play may continue for a stated period. If the position then remains essentially unchanged, the referee can order the restart.

A re-rack has an unusual effect on the game state:

$$
(\text{current score},\ \text{current balls})
\rightarrow
(0\text{–}0,\ \text{opening position})
$$

The accumulated scores in the frame are nullified and all balls are reset as at the start of the frame.

The frame itself, however, has not been won or lost. It is restarted, with the same player making the opening stroke.

So a re-rack is neither a normal continuing transition nor a terminal state.

It is a **reset transition**:

$$
S_t
\rightarrow
S_0
$$

where the history of the frame remains part of the match record, but the playable state is returned to the opening position.

A special version can also occur during a respotted black. In that case only the black is respotted and the same player makes the opening stroke again.

[WPBSA Rules — Section 3, Rule 17: Stalemate (p. 33)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=37)

## A re-rack requires information about the frame history

There is a problem if we try to implement the stalemate rule using our current `SnookerState`.

The rules require the **same player who made the original opening stroke** to make the opening stroke after a re-rack.

But our current state records only:

$$
\text{player to act}
$$

It does not record who originally started the frame.

Those are not necessarily the same player when a stalemate occurs.

So the transition:

$$
S_t
\rightarrow
S_0
$$

cannot be determined completely from our current state.

We need one additional piece of information:

$$
\text{opening player}
$$

Our richer state would therefore be:

$$
S =
(
\text{score}_A,
\text{score}_B,
\text{reds remaining},
\text{colours remaining},
\text{player to act},
\text{opening player},
\text{ball on},
\text{free ball available},
\text{nominated free ball}
)
$$

This is a different reason for expanding the state from the one we encountered with a free ball.

The free-ball rule required more information about the **current position**.

The stalemate rule requires information about the **history of the frame**.

So a sufficient game state must contain not only everything needed to describe what is happening now, but also any past information that the rules can later use to determine what happens next.

In [17]:
from dataclasses import dataclass, replace
from typing import Tuple


# Extend the state one more time so that the identity of the
# player who opened the frame is preserved.
#
# This information normally has no effect on play, but becomes
# necessary if the frame is restarted after a stalemate.
@dataclass(frozen=True)
class SnookerState:
    score_a: int
    score_b: int
    reds_remaining: int
    colours_remaining: Tuple[str, ...]
    player_to_act: str
    ball_on: str
    opening_player: str = "A"
    free_ball_available: bool = False
    nominated_free_ball: str | None = None


def rerack_frame(state: SnookerState) -> SnookerState:
    """
    Return the opening state after a full-frame re-rack.

    A re-rack:
    - resets both scores to zero;
    - restores all 15 reds and all six colours;
    - makes a red the ball on;
    - clears any free-ball state;
    - gives the opening stroke to the same player who
      originally opened the frame.
    """

    # The recorded opening player must be one of the two players.
    if state.opening_player not in ("A", "B"):
        raise ValueError("Opening player must be 'A' or 'B'.")

    # Return the frame to its opening playable state while
    # preserving the historical fact of who opened the frame.
    return SnookerState(
        score_a=0,
        score_b=0,
        reds_remaining=15,
        colours_remaining=(
            "yellow",
            "green",
            "brown",
            "blue",
            "pink",
            "black",
        ),
        player_to_act=state.opening_player,
        ball_on="red",
        opening_player=state.opening_player,
        free_ball_available=False,
        nominated_free_ball=None,
    )


# Construct an illustrative stalemate position.
#
# Player B happens to be at the table when the re-rack is ordered,
# but Player A made the original opening stroke.
stalemate_position = SnookerState(
    score_a=12,
    score_b=8,
    reds_remaining=13,
    colours_remaining=(
        "yellow",
        "green",
        "brown",
        "blue",
        "pink",
        "black",
    ),
    player_to_act="B",
    ball_on="red",
    opening_player="A",
)

# Restart the frame.
after_rerack = rerack_frame(stalemate_position)

# Compare the state before and after the reset.
print("Before re-rack:")
print(stalemate_position)

print("\nAfter re-rack:")
print(after_rerack)

Before re-rack:
SnookerState(score_a=12, score_b=8, reds_remaining=13, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='B', ball_on='red', opening_player='A', free_ball_available=False, nominated_free_ball=None)

After re-rack:
SnookerState(score_a=0, score_b=0, reds_remaining=15, colours_remaining=('yellow', 'green', 'brown', 'blue', 'pink', 'black'), player_to_act='A', ball_on='red', opening_player='A', free_ball_available=False, nominated_free_ball=None)


## Re-rack as a reset rather than a new frame

The stalemate example produces a transition unlike any we have seen so far.

Before the re-rack:

$$
12\text{–}8
$$

with 13 reds remaining and Player B at the table.

After the re-rack:

$$
0\text{–}0
$$

with all 15 reds restored and Player A to play.

So several parts of the state have been reset simultaneously:

- both scores return to zero;
- all balls return to their opening configuration;
- a red becomes the ball on;
- the player who originally opened the frame plays first again.

But the frame has **not** become a different frame in the match.

This means that a re-rack is best understood as a reset of the playable state rather than a terminal transition:

$$
S_t
\rightarrow
S_0
$$

while some information from the history of the frame must still be preserved.

The example also reinforces the distinction between a **game tree** and a **state graph**.

The sequence of events leading to the stalemate cannot be undone, so the history still forms a forward-moving branch.

But the resulting playable state can return to one that is effectively identical to the beginning of the frame.

Snooker therefore contains:

- forward transitions;
- repeating states;
- terminal states;
- and reset transitions.

Even with a comparatively compact rule set, the structure is already richer than a simple sequence of independent shots.

## The simplified game tree so far

We can now describe a snooker frame as a collection of states connected by legal transitions.

A simplified frame state contains:

$$
S =
(
\text{score}_A,
\text{score}_B,
\text{reds remaining},
\text{colours remaining},
\text{player to act},
\text{opening player},
\text{ball on},
\text{free ball available},
\text{nominated free ball}
)
$$

From a given state, several different kinds of transition may be possible.

### Scoring transitions

A legal pot changes the score and may also change:

- the number of balls remaining;
- the ball on;
- whether the visit continues;
- and whether the frame has reached a terminal state.

### Non-scoring transitions

A legal non-scoring shot can leave the score and balls unchanged while transferring control to the opponent.

### Foul transitions

A foul can:

- award penalty points;
- alter who plays next;
- create a free-ball situation;
- allow the offender to be required to play again;
- or, after a foul and a miss, allow an earlier table position to be restored.

### Player-choice transitions

Some branches exist because a player is given a choice under the rules, such as:

- which colour to nominate;
- which ball to nominate as a free ball;
- whether to require the opponent to play again;
- whether to accept a valid concession;
- or, after the draw of lots for a respotted black, whether to play first.

### Rule and referee transitions

Other branches are created by the rules or by a referee's ruling rather than by a player's shot choice.

Examples include:

- a free-ball call;
- a foul-and-a-miss ruling;
- a respotted black;
- a stalemate ruling;
- and the determination that a frame has ended.

### Repeating transitions

Some legal shots can return the frame to the same type of state.

A safety exchange on a respotted black may repeatedly alternate:

$$
A
\rightarrow
B
\rightarrow
A
\rightarrow
B
\rightarrow
\cdots
$$

without changing either the score or the balls remaining.

### Reset transitions

A stalemate can produce a re-rack:

$$
S_t
\rightarrow
S_0
$$

The score and playable table state are reset, while historical information such as the original opening player must be preserved.

### Terminal transitions

A frame can end through several different routes:

- completion of the normal scoring sequence;
- a pot or foul on a respotted black;
- an accepted valid concession;
- or another rule-defined terminal condition.

So even in our simplified representation, a frame is not merely a sequence of pots and misses.

It is a sequence of **states, actions, rulings, choices, repeated states, resets and terminal conditions**.

The next question is how much of the real game this simplified state still leaves out.

## The missing state: where the balls actually are

Our `SnookerState` now contains enough information to represent many of the rule-driven transitions in a frame.

But it still leaves out the most important information needed to model actual play:

> **the physical positions of the balls on the table**

Two frames can have exactly the same simplified state:

$$
(
40,\ 32,\ 6\text{ reds},\ A\text{ to act},\ \text{red on}
)
$$

while presenting completely different problems to the player.

In one position:

- a red may be directly available over a pocket;
- the cue-ball may already be well positioned for the black;
- the remaining reds may be spread around the table.

In another:

- no red may be directly pottable;
- the cue-ball may be tight against a cushion;
- several reds may be tied together;
- or the player may be snookered.

Our simplified state treats these positions as identical, even though their available shots and likely outcomes could be very different.

A fuller representation would therefore need a physical table state such as:

$$
T =
(
\text{cue-ball position},
\text{positions of remaining reds},
\text{positions of colours}
)
$$

and the complete game state would become something closer to:

$$
G = (S, T)
$$

where:

- $S$ describes the **rule and scoring state**;
- $T$ describes the **physical configuration of the table**.

This missing information is not merely cosmetic.

Without the physical table state, our model cannot determine:

- whether a player is snookered;
- whether a free ball should be called;
- which legal shots are actually available;
- how difficult those shots are;
- whether balls can be replaced after a foul and a miss;
- or what position will be left after a particular stroke.

So the structure we have built is best understood as a model of the **rules governing a frame**, rather than a complete model of snooker play.

## Adding the physical table state

Our `SnookerState` describes the rule and scoring state of a frame, but it does not yet describe where the balls actually are.

Representing that physical position is relatively straightforward.

At any resting point between strokes, each ball on the table can be described by its coordinates:

$$
T =
{
(\text{ball}_1, x_1, y_1),
(\text{ball}_2, x_2, y_2),
\ldots
}
$$

Not every arbitrary set of coordinates represents a physically possible position. Balls cannot occupy the same space or overlap, so the centres of any two balls must be separated by at least one ball diameter. Ball centres must also remain within the playable area of the table, with further constraints introduced by the cushions and pocket openings.

These are geometric validity constraints on $T$ rather than additional parts of the state itself, and can be checked whenever a table position is created.

The dimensions of the table, positions of the pockets and fixed spots for the colours are properties of the game itself, so they do not need to be stored separately in every state.

Combining the rule state and table state gives:

$$
G = (S, T)
$$

where:

* $S$ describes the score, balls remaining, player to act and other rule information;
* $T$ describes the coordinates of every ball currently on the table.

From those coordinates, other features can in principle be calculated rather than stored directly.

For example:

* whether the cue-ball has a clear line to a ball;
* whether the striker is snookered;
* distances between balls and pockets;
* potting angles;
* which balls obstruct others;
* whether balls are touching;
* and how tightly groups of reds are clustered.

So representing a snooker position is not especially difficult.

The harder problem is describing a **stroke** and determining the next table position that results from it.

A stroke would need additional information such as direction, speed, contact and spin, and its outcome would depend on both player execution and the physics of the balls.

This gives us another useful separation:

$$
\text{game state}
+
\text{chosen stroke}
\rightarrow
\text{next game state}
$$

The state itself can be represented compactly. The difficult modelling problem is the transition between physical states.


In [18]:
from dataclasses import dataclass
from math import hypot
from typing import Tuple


# Official table and ball dimensions in millimetres.
TABLE_LENGTH_MM = 3569.0
TABLE_WIDTH_MM = 1778.0
BALL_DIAMETER_MM = 52.5
BALL_RADIUS_MM = BALL_DIAMETER_MM / 2


# Represent one ball resting on the table.
@dataclass(frozen=True)
class BallPosition:
    ball: str
    x: float
    y: float


@dataclass(frozen=True)
class TableState:
    balls: Tuple[BallPosition, ...]


def validate_table_state(table: TableState) -> None:
    """
    Check basic geometric validity of a resting table position.

    This simplified validation requires:
    - each ball centre to lie within the playing area;
    - no two balls to overlap.

    Exact cushion and pocket-jaw geometry is not yet modelled.
    """

    # Check that every ball centre lies within the rectangular
    # playing area, allowing for the radius of the ball.
    for ball in table.balls:
        if not (
            BALL_RADIUS_MM <= ball.x <= TABLE_LENGTH_MM - BALL_RADIUS_MM
            and BALL_RADIUS_MM <= ball.y <= TABLE_WIDTH_MM - BALL_RADIUS_MM
        ):
            raise ValueError(
                f"{ball.ball} lies outside the valid playing area."
            )

    # Check every pair of balls.
    #
    # Their centres must be separated by at least one full
    # ball diameter; otherwise the balls would overlap.
    for i, ball_a in enumerate(table.balls):
        for ball_b in table.balls[i + 1:]:
            distance = hypot(
                ball_a.x - ball_b.x,
                ball_a.y - ball_b.y,
            )

            if distance < BALL_DIAMETER_MM:
                raise ValueError(
                    f"{ball_a.ball} overlaps {ball_b.ball}."
                )


# Construct a small illustrative legal position.
table_position = TableState(
    balls=(
        BallPosition("cue", 800, 850),
        BallPosition("red_01", 2200, 850),
        BallPosition("red_02", 2280, 790),
        BallPosition("red_03", 2280, 910),
        BallPosition("yellow", 737, 500),
        BallPosition("green", 737, 1278),
        BallPosition("brown", 737, 889),
        BallPosition("blue", 1784.5, 889),
        BallPosition("pink", 2600, 889),
        BallPosition("black", 3245, 889),
    )
)

# Verify that the position satisfies our basic geometric constraints.
validate_table_state(table_position)

table_position

TableState(balls=(BallPosition(ball='cue', x=800, y=850), BallPosition(ball='red_01', x=2200, y=850), BallPosition(ball='red_02', x=2280, y=790), BallPosition(ball='red_03', x=2280, y=910), BallPosition(ball='yellow', x=737, y=500), BallPosition(ball='green', x=737, y=1278), BallPosition(ball='brown', x=737, y=889), BallPosition(ball='blue', x=1784.5, y=889), BallPosition(ball='pink', x=2600, y=889), BallPosition(ball='black', x=3245, y=889)))

## The table state is constrained

The example position can be stored compactly as a collection of ball identities and coordinates.

But the coordinates cannot take arbitrary values.

For a table state to be physically possible, at minimum:

- every ball centre must lie within the playable area;
- two balls cannot occupy the same space;
- the distance between the centres of two balls cannot be less than one ball diameter.

For any two balls \(i\) and \(j\), this gives the constraint:

$$
\sqrt{(x_i-x_j)^2 + (y_i-y_j)^2}
\geq
d
$$

where \(d\) is the diameter of a snooker ball.

So the physical state space is not simply every possible collection of coordinates:

$$
T \neq \mathbb{R}^{2n}
$$

Instead, only a subset of those coordinate combinations represents physically valid table positions.

Our current validator captures the most basic constraints. A more exact representation could later include the detailed geometry of the cushions, pocket openings and jaws.

The important point for now is that these are **constraints on the table state**, rather than extra information that has to be stored separately in every position.

The next step is to check that our representation rejects a position that cannot physically exist.

In [19]:
# Construct an impossible table state.
#
# The cue-ball and red_01 are only 30 mm apart between centres,
# which is less than the 52.5 mm diameter of a snooker ball.
# The two balls would therefore overlap physically.
invalid_table_position = TableState(
    balls=(
        BallPosition("cue", 800, 850),
        BallPosition("red_01", 830, 850),
        BallPosition("blue", 1784.5, 889),
        BallPosition("black", 3245, 889),
    )
)


# The validator should reject the position rather than
# accepting it as a possible game state.
try:
    validate_table_state(invalid_table_position)
    print("Position accepted.")
except ValueError as error:
    print("Position rejected:", error)

Position rejected: cue overlaps red_01.


## Where the observable state ends

We can represent the rule state of a frame and, in principle, the physical table state:

$$
G = (S, T)
$$

where:

- $S$ contains the score, balls remaining, player to act and other rule information;
- $T$ contains the positions of the balls on the table.

The next state depends on the stroke that is played.

But detailed stroke information — such as cue direction, speed, spin and exact contact — is not normally recorded in match data.

So although those variables matter physically, they are not necessarily useful as part of a practical statistical model.

For observational analysis, the useful state is therefore likely to be the information that can actually be measured or reconstructed:

- score;
- balls remaining;
- player at the table;
- ball on;
- break or visit information;
- and, where available, ball positions.

The transition from one observed state to the next can then be treated as the result of the player's action without requiring a complete physical simulation of the stroke.

This gives us a practical boundary:

$$
\text{observed game state}
\rightarrow
\text{observed next game state}
$$

rather than:

$$
\text{cue mechanics}
\rightarrow
\text{physics simulation}
\rightarrow
\text{next state}
$$

For the purposes of analysing real snooker, the first representation is likely to be much more useful.

## From frames to a match

Once the internal structure of a frame is collapsed to a winner, the next level of snooker becomes much simpler.

Under the official rules, a game can be decided by winning the required number of frames.

In the usual professional format, this is described as a **best-of-(n)** match.

For example, in a best-of-7:

$$
\text{frames required to win}
=============================

# \left\lfloor \frac{7}{2} \right\rfloor + 1

4
$$

So the match ends as soon as either player reaches four frame wins.

It does not necessarily contain all seven frames:

$$
4\text{–}0
\rightarrow
\text{match ends}
$$

just as:

$$
4\text{–}3
\rightarrow
\text{match ends}
$$

At this level, the state can be represented very compactly:

$$
M =
(
\text{frames won}_A,
\text{frames won}_B,
\text{frames required}
)
$$

and each completed frame produces one of only two ordinary transitions:

$$
M_t
\rightarrow
\begin{cases}
\text{A wins frame} \
\text{B wins frame}
\end{cases}
$$

The player who makes the opening stroke alternates from one frame to the next.

So there is a striking change in complexity between the two levels:

$$
\text{complex internal frame}
\rightarrow
\text{binary frame result}
\rightarrow
\text{simple match state}
$$

Once we move to the match level, all of the internal structure of a frame can be compressed into a single outcome:

$$
\text{Player A wins frame}
\quad\text{or}\quad
\text{Player B wins frame}
$$

That abstraction makes match-level modelling much simpler, but it also hides everything that happened within the frame to produce the result.

[WPBSA Rules — Section 3, Rule 1: Description (p. 17)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=21)

[WPBSA Rules — Section 3, Rule 3: Mode of Play (p. 18)](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf#page=22)


In [20]:
# Represent the state of a snooker match.
#
# Once individual frames are reduced to a winner,
# the match state becomes much simpler than the frame state.
@dataclass(frozen=True)
class MatchState:
    frames_a: int
    frames_b: int
    frames_required: int
    opening_player_next_frame: str


def record_frame_winner(
    state: MatchState,
    winner: str,
) -> MatchState:
    """
    Return the next match state after a frame has been completed.

    This transition:
    - adds one frame to the winner;
    - alternates the opening player for the next frame;
    - leaves the target number of frames unchanged.
    """

    # Only the two players in the match can win a frame.
    if winner not in ("A", "B"):
        raise ValueError("Winner must be 'A' or 'B'.")

    # A completed match cannot accept another frame result.
    if (
        state.frames_a >= state.frames_required
        or state.frames_b >= state.frames_required
    ):
        raise ValueError("The match has already ended.")

    # Update the frame score.
    frames_a = state.frames_a
    frames_b = state.frames_b

    if winner == "A":
        frames_a += 1
    else:
        frames_b += 1

    # The opening stroke alternates from one frame to the next.
    next_opening_player = (
        "B"
        if state.opening_player_next_frame == "A"
        else "A"
    )

    # Return the new match state.
    return MatchState(
        frames_a=frames_a,
        frames_b=frames_b,
        frames_required=state.frames_required,
        opening_player_next_frame=next_opening_player,
    )


# Start a best-of-7 match.
#
# Four frames are required to win,
# and Player A will open the first frame.
match_start = MatchState(
    frames_a=0,
    frames_b=0,
    frames_required=4,
    opening_player_next_frame="A",
)

# Suppose Player A wins the opening frame.
after_frame_one = record_frame_winner(
    match_start,
    winner="A",
)

# Inspect the new match state.
after_frame_one

MatchState(frames_a=1, frames_b=0, frames_required=4, opening_player_next_frame='B')

## The match tree is much smaller

At the match level, each completed frame reduces to a binary result:

$$
\text{frame}
\rightarrow
\begin{cases}
A \text{ wins} \\
B \text{ wins}
\end{cases}
$$

Starting from:

$$
0\text{–}0
$$

a best-of-7 match can therefore branch through states such as:

$$
0\text{–}0
\rightarrow
1\text{–}0
\rightarrow
1\text{–}1
\rightarrow
2\text{–}1
\rightarrow
\cdots
$$

But the tree stops as soon as either player reaches four frame wins.

So states such as:

$$
4\text{–}0,\quad
4\text{–}1,\quad
4\text{–}2,\quad
4\text{–}3
$$

are terminal states for Player A, with the corresponding reversed scores terminal for Player B.

This makes the match tree comparatively small and regular.

The complicated structure lies inside each frame. Once a frame has been completed, all of that internal detail can be replaced by one binary result before the match moves to its next state.

The match format therefore creates a natural hierarchy:

$$
\text{shot outcomes}
\rightarrow
\text{frame states}
\rightarrow
\text{frame winner}
\rightarrow
\text{match state}
\rightarrow
\text{match winner}
$$

Each level retains only the information needed by the level above it.

In [21]:
def match_winner(state: MatchState) -> str | None:
    """
    Return the winner of the match if the target number
    of frames has been reached.

    If the match is still live, return None.
    """

    # Player A wins as soon as they reach the required
    # number of frame wins.
    if state.frames_a >= state.frames_required:
        return "A"

    # Player B wins under the equivalent condition.
    if state.frames_b >= state.frames_required:
        return "B"

    # Otherwise the match is still in progress.
    return None


# Build one complete possible path through a best-of-7 match.
#
# Player A will win the match 4-2.
match_path = [match_start]

for winner in ("A", "B", "A", "A", "B", "A"):
    next_state = record_frame_winner(
        match_path[-1],
        winner=winner,
    )
    match_path.append(next_state)


# Display each successive match state and whether
# the match has become terminal.
for frame_number, state in enumerate(match_path):
    print(
        f"After {frame_number} frames:",
        state,
        "| Winner:",
        match_winner(state),
    )

After 0 frames: MatchState(frames_a=0, frames_b=0, frames_required=4, opening_player_next_frame='A') | Winner: None
After 1 frames: MatchState(frames_a=1, frames_b=0, frames_required=4, opening_player_next_frame='B') | Winner: None
After 2 frames: MatchState(frames_a=1, frames_b=1, frames_required=4, opening_player_next_frame='A') | Winner: None
After 3 frames: MatchState(frames_a=2, frames_b=1, frames_required=4, opening_player_next_frame='B') | Winner: None
After 4 frames: MatchState(frames_a=3, frames_b=1, frames_required=4, opening_player_next_frame='A') | Winner: None
After 5 frames: MatchState(frames_a=3, frames_b=2, frames_required=4, opening_player_next_frame='B') | Winner: None
After 6 frames: MatchState(frames_a=4, frames_b=2, frames_required=4, opening_player_next_frame='A') | Winner: A


## How large is the match tree?

Because each completed frame has only two possible winners, the match-level tree can be counted exactly.

Consider a best-of-7 match, where the first player to four frames wins.

A match can finish:

- 4–0 after 4 frames;
- 4–1 after 5 frames;
- 4–2 after 6 frames;
- 4–3 after 7 frames;

or with the corresponding scores reversed.

The number of possible **histories** is larger than the number of possible scores.

For example, a 4–2 result can arise through many different sequences of frame winners:

$$
A,A,B,A,B,A
$$

and:

$$
B,A,A,B,A,A
$$

both finish at the same terminal state:

$$
4\text{–}2
$$

So once again we need to distinguish between:

- the **game tree**, which records every possible sequence of frame results;
- the **state graph**, in which many different sequences merge into the same match score.

At match level, both are small enough that we can enumerate them completely.

The next step is to count the possible states and complete frame-result histories for a best-of-7 match.

In [22]:
from math import comb


def count_match_states(frames_required: int) -> tuple[int, int, int]:
    """
    Count the reachable score states in a first-to-N match.

    Returns:
    - number of live states;
    - number of terminal states;
    - total number of reachable states.
    """

    # A live match has both players below the winning target.
    #
    # Each player can therefore have:
    # 0, 1, ..., frames_required - 1
    live_states = frames_required ** 2

    # A terminal score has exactly one player on the target
    # and the loser somewhere from 0 to frames_required - 1.
    #
    # Either player can be the winner.
    terminal_states = 2 * frames_required

    return (
        live_states,
        terminal_states,
        live_states + terminal_states,
    )


def count_complete_histories(frames_required: int) -> int:
    """
    Count every possible complete sequence of frame winners
    in a first-to-N match.

    For a final score N-k:
    - the final frame must be won by the match winner;
    - before that frame, the winner must have N-1 frame wins;
    - the loser must have k frame wins.

    The possible orderings of those earlier results are
    counted using a binomial coefficient.
    """

    histories_for_one_winner = 0

    # The losing player can finish with anything from
    # zero to N-1 frame wins.
    for loser_frames in range(frames_required):

        # Before the final frame there are:
        #
        #   (N - 1) winner frames + loser_frames
        #
        # and we choose where the loser's frame wins occur.
        histories = comb(
            (frames_required - 1) + loser_frames,
            loser_frames,
        )

        histories_for_one_winner += histories

    # Either Player A or Player B can win the match.
    return 2 * histories_for_one_winner


# A best-of-7 is first to four frames.
frames_required = 4

live_states, terminal_states, total_states = count_match_states(
    frames_required
)

complete_histories = count_complete_histories(
    frames_required
)

print("Live score states:", live_states)
print("Terminal score states:", terminal_states)
print("Total reachable score states:", total_states)
print("Complete frame-result histories:", complete_histories)

Live score states: 16
Terminal score states: 8
Total reachable score states: 24
Complete frame-result histories: 70


## Many histories collapse into very few match states

A best-of-7 match has only:

- **16 live score states**;
- **8 terminal score states**;
- **24 reachable score states in total**.

But there are:

$$
70
$$

different complete sequences of frame winners that can lead from 0–0 to a finished match.

The difference comes from different histories merging into the same state.

For example:

$$
A,B,A
$$

and:

$$
B,A,A
$$

both produce the same score:

$$
2\text{–}1
$$

Once the current match score is known, the order in which those earlier frame wins occurred is not needed to determine how many more frames are required to win.

So at match level there is a substantial compression:

$$
70\text{ complete histories}
\rightarrow
24\text{ reachable score states}
$$

This is one reason the format of a snooker match is comparatively simple to model.

The internal structure of each frame may contain many different states and paths, but the match itself retains very little of that information: essentially only the accumulated number of frame wins.

The next question is how this match-level structure grows as the required number of frames increases.

In [23]:
# Compare the size of the match-level state space and history tree
# across several common best-of match lengths.
#
# For a best-of-(2N - 1) match, the first player to N frames wins.
match_formats = (7, 11, 19, 35)

format_results = []

for best_of in match_formats:

    # Convert the advertised best-of length into the
    # number of frames required to win the match.
    frames_required = (best_of // 2) + 1

    # Count the distinct reachable score states.
    live_states, terminal_states, total_states = count_match_states(
        frames_required
    )

    # Count every possible complete sequence of frame winners.
    complete_histories = count_complete_histories(
        frames_required
    )

    format_results.append(
        (
            best_of,
            frames_required,
            live_states,
            terminal_states,
            total_states,
            complete_histories,
        )
    )


# Display the comparison.
print(
    f"{'Format':<12}"
    f"{'First to':<12}"
    f"{'Live':<10}"
    f"{'Terminal':<12}"
    f"{'Total states':<15}"
    f"{'Histories':<15}"
)

for (
    best_of,
    frames_required,
    live_states,
    terminal_states,
    total_states,
    complete_histories,
) in format_results:
    print(
        f"{'Best of ' + str(best_of):<12}"
        f"{frames_required:<12}"
        f"{live_states:<10}"
        f"{terminal_states:<12}"
        f"{total_states:<15}"
        f"{complete_histories:<15}"
    )

Format      First to    Live      Terminal    Total states   Histories      
Best of 7   4           16        8           24             70             
Best of 11  6           36        12          48             924            
Best of 19  10          100       20          120            184756         
Best of 35  18          324       36          360            9075135300     


## The opening player alternates between frames

Although a completed frame can be reduced to a binary result, one piece of information carries from the match format into the next frame: who makes the opening stroke.

The opening player alternates from frame to frame.

So if Player A opens frame 1, a best-of-7 has the sequence:

$$
A,\ B,\ A,\ B,\ A,\ B,\ A
$$

This has a small but potentially important consequence.

Because a best-of match contains an odd maximum number of frames, the player who opens the first frame also opens the deciding frame if the match reaches its full distance.

For a best-of-7:

$$
\text{frame 1}: A
$$

and:

$$
\text{frame 7}: A
$$

The same is true for any ordinary odd-length best-of format.

This does not imply that opening a frame is an advantage. That is a separate empirical question.

But if opening the frame does affect the probability of winning it, then the alternating break-off rule becomes part of the match model rather than something that can simply be ignored.

So a more complete match state may need to retain:

$$
\text{opening player next frame}
$$

alongside the current frame score.

In [24]:
def opening_sequence(
    best_of: int,
    first_opening_player: str = "A",
) -> list[str]:
    """
    Return the opening-player sequence for an odd-length
    best-of snooker match.

    The opening stroke alternates from frame to frame.
    """

    # Standard best-of formats contain an odd maximum
    # number of frames.
    if best_of <= 0 or best_of % 2 == 0:
        raise ValueError("best_of must be a positive odd number.")

    # Only the two players in the match can open a frame.
    if first_opening_player not in ("A", "B"):
        raise ValueError("First opening player must be 'A' or 'B'.")

    # Identify the other player so we can alternate
    # the opening stroke from frame to frame.
    other_player = (
        "B"
        if first_opening_player == "A"
        else "A"
    )

    # Odd-numbered frames are opened by the player who
    # opened frame 1; even-numbered frames by the opponent.
    return [
        first_opening_player if frame % 2 == 1 else other_player
        for frame in range(1, best_of + 1)
    ]


# Show the complete opening sequence for a best-of-7 match.
best_of_7_openers = opening_sequence(
    best_of=7,
    first_opening_player="A",
)

print("Best-of-7 opening sequence:", best_of_7_openers)
print("Opening player in frame 1:", best_of_7_openers[0])
print("Opening player in frame 7:", best_of_7_openers[-1])

Best-of-7 opening sequence: ['A', 'B', 'A', 'B', 'A', 'B', 'A']
Opening player in frame 1: A
Opening player in frame 7: A


## Frame scores do not carry into the match score

The score accumulated inside a frame is used only to determine the winner of that frame.

Once the frame ends, those points disappear from the state of the match.

For example, these two frame results:

$$
120\text{–}0
$$

and:

$$
65\text{–}64
$$

have exactly the same effect at match level:

$$
\text{one frame win}
$$

So if Player A wins either frame, the match state changes in the same way:

$$
(0,0)
\rightarrow
(1,0)
$$

The size of the frame victory is not carried forward.

This creates a strong boundary between the two levels of the game:

$$
\text{frame points}
\rightarrow
\text{frame winner}
\rightarrow
\text{match score}
$$

Only the identity of the frame winner survives.

A player can therefore score many more points than their opponent across an entire match and still lose the match if those points are distributed across the frames less effectively.

This is another reason the match state can remain so compact: it does not need to retain the scores, breaks or margins from completed frames in order to apply the rules of the remaining match.

## The hierarchy of a snooker match

We can now describe snooker at several nested levels:

$$
\text{match}
\rightarrow
\text{frame}
\rightarrow
\text{visit}
\rightarrow
\text{stroke}
$$

Each level has its own state and its own terminal condition.

### Stroke level

A stroke acts on the current physical and rule state of the table.

Its result may:

- score points;
- remove or respot a ball;
- continue the visit;
- end the visit;
- produce a foul;
- or trigger another rule-dependent branch.

### Visit level

A visit continues while the same player retains control of the table.

It ends when control passes to the opponent, although the rules can sometimes require the same player to play again after a foul.

### Frame level

The frame combines:

- the accumulated points;
- the balls remaining;
- the player to act;
- the ball on;
- the physical table position;
- and any additional rule state required by situations such as a free ball.

A frame can contain continuing states, repeating states, reset transitions and terminal states.

Once the frame ends, however, almost all of that information can be discarded at match level.

The frame contributes only:

$$
\text{winner} \in \{A,B\}
$$

### Match level

The match retains the number of frames won by each player and the information needed to determine the next frame.

The basic progression is therefore:

$$
\text{stroke outcome}
\rightarrow
\text{visit}
\rightarrow
\text{frame state}
\rightarrow
\text{frame winner}
\rightarrow
\text{match state}
\rightarrow
\text{match winner}
$$

Information is progressively compressed as we move up the hierarchy.

The exact score, table position and sequence of shots matter within a frame, but once that frame is complete they are not required by the rules to determine how the match proceeds.

This separation between levels is one of the defining structural features of snooker.

## The minimum state at each level

The hierarchy also lets us ask a useful modelling question:

> **What is the minimum information needed to determine the legal continuations from the current position?**

Anything that does not affect what can happen next does not have to be part of the state.

### Match state

At match level, very little information is required:

$$
M =
(
\text{frames won}_A,
\text{frames won}_B,
\text{frames required},
\text{opening player next frame}
)
$$

The points scored in previous frames, winning margins and sequence of earlier shots are not required by the rules to continue the match.

### Frame rule state

Within a frame, more information must be retained:

$$
S =
(
\text{score}_A,
\text{score}_B,
\text{reds remaining},
\text{colours remaining},
\text{player to act},
\text{opening player},
\text{ball on},
\text{free-ball state}
)
$$

The opening player is normally irrelevant during play, but must be retained because a re-rack can make it relevant again.

Similarly, free-ball information is normally absent but becomes necessary when that rule is invoked.

### Physical table state

If we need to determine which shots are actually available, the rule state is not enough.

We also need:

$$
T =
{
(\text{ball},x,y)
}
$$

for every ball currently on the table.

The combined frame state is therefore:

$$
G = (S,T)
$$

### Visit state

A separate visit state is not necessarily required to continue the game.

At a resting point between strokes, `player_to_act` already tells us whose visit is in progress. Information such as the current break score may be useful for analysis and record keeping, but it is not normally required to determine the next legal state.

This distinction is useful:

$$
\text{information worth recording}
\neq
\text{information required by the game state}
$$

A model may retain much more information than the minimum state if that information helps answer a particular analytical question.

But the underlying game can be represented more compactly by keeping only the information that can affect future legal transitions.


## Snooker as a state-transition system

The structure developed in this study can be expressed quite simply.

At any point in a frame there is a current game state:

$$
G_t
$$

The rules determine a set of legal actions or events from that state:

$$
A(G_t)
$$

and each realised action produces another state:

$$
G_t
\rightarrow
G_{t+1}
$$

This continues until a terminal state is reached and the frame winner is determined.

Some transitions are produced directly by a player's stroke:

$$
\text{pot}
\rightarrow
\text{new score and table state}
$$

Others depend on a player's subsequent choice:

$$
\text{foul}
\rightarrow
\text{require offender to play again}
$$

Some depend on a referee's ruling:

$$
\text{foul}
\rightarrow
\text{free ball}
$$

and some are imposed entirely by the format of the game:

$$
\text{frame winner}
\rightarrow
\text{updated match score}
$$

The resulting structure is therefore more accurately described as a **state-transition graph** than as a simple tree of unique states.

The history of play branches like a tree because every frame has one particular sequence of events.

But the underlying state graph can contain:

* different histories that reach the same state;
* repeated states during safety exchanges;
* reset transitions following a re-rack;
* and terminal states reached through several different routes.

Despite these possibilities, the rule structure remains comparatively compact.

The complexity of actual snooker comes less from storing the state than from the enormous range of physical positions and shots that can occur within that state-transition system.


## What this means for modelling snooker

The rules of snooker create a useful hierarchy of possible models.

At the most detailed level, a model could attempt to represent:

$$
\text{ball positions}
+
\text{legal actions}
+
\text{player execution}
\rightarrow
\text{next table position}
$$

Such a model would need detailed positional data and some representation of shot difficulty and execution.

A simpler model can ignore the mechanics of individual strokes and work with observed changes in the frame state:

$$
G_t
\rightarrow
G_{t+1}
$$

This retains information such as the score, balls remaining and player at the table without attempting to reproduce the physics that produced each transition.

At a still higher level, the entire internal structure of a frame can be collapsed into:

$$
\text{Player A wins frame}
\quad\text{or}\quad
\text{Player B wins frame}
$$

and a match can then be represented purely as a sequence of frame results.

None of these representations is inherently the correct model of snooker.

They answer different questions.

A shot-level model might be appropriate for studying position, safety play or shot selection.

A frame-state model might be useful for questions such as comeback probability, the value of a particular lead or how winning chances change as balls disappear.

A frame-result model may be sufficient when the question concerns match format or the probability of winning a match.

The important choice is therefore not simply how detailed a model can be.

It is:

> **What information must be retained to answer the question being asked?**

Snooker is particularly convenient in this respect because the rules create clear boundaries between strokes, visits, frames and matches.

That allows information to be discarded deliberately as the level of analysis moves upward, rather than accidentally hiding assumptions inside the model.


## Conclusion

Snooker can be represented as a relatively compact hierarchy of states and transitions.

At the match level, the structure is especially simple:

$$
\text{frame winner}
\rightarrow
\text{updated match score}
$$

Within a frame, more information is required. The state must retain the score, balls remaining, player to act, ball on and any additional information that can affect future legal play.

The rules introduce several distinct kinds of transition:

* ordinary scoring and non-scoring strokes;
* fouls and penalty points;
* player choices following fouls;
* free balls;
* respotted blacks;
* concessions;
* foul-and-a-miss replacements;
* stalemates and re-racks;
* and terminal frame conditions.

Some histories can return to the same rule state, while a re-rack can reset much of the playable state altogether. For that reason, the structure is better thought of as a **state-transition graph** than simply a tree of unique states.

The physical table position can also be represented compactly as the coordinates of the balls:

$$
T =
{
(\text{ball},x,y)
}
$$

subject to the geometric constraints of the table and balls.

Combining this with the rule state gives:

$$
G=(S,T)
$$

The difficult part is not necessarily representing a snooker position. It is modelling the transition from one physical position to the next: the shot selected, how well it is executed and the resulting movement of the balls.

This gives snooker a useful layered structure:

$$
\text{physical position}
\rightarrow
\text{stroke outcome}
\rightarrow
\text{frame state}
\rightarrow
\text{frame winner}
\rightarrow
\text{match state}
\rightarrow
\text{match winner}
$$

A model does not need to preserve every layer.

The appropriate representation depends on the question being asked, and information can be deliberately discarded as the analysis moves from the detailed structure of a frame towards the much simpler structure of a match.


## Glossary

**Ball on**
Any ball which may be legally struck first by the cue-ball, or any ball which may be legally potted, according to the current state of play.

**Break**
The number of points scored by a player during one turn at the table.

**Concession**
An offer to concede the frame. Under the rules, a player must not concede while neither player requires penalty points.

**Foul**
A stroke or action that infringes the rules and normally results in penalty points being awarded to the opponent.

**Foul and a miss**
A referee's call made in specified circumstances following a foul in which the striker has failed to hit the ball on first.

**Frame**
One unit of play in which the players score points until the frame is completed under the rules.

**Free ball**
A ball nominated by the incoming player following a foul when the cue-ball has been left snookered on all balls on. The nominated ball is treated as the ball on for that stroke.

**Penalty points**
Points awarded to the non-offending player following a foul.

**Re-rack**
The common term for restarting a frame after a stalemate, with the balls reset and the frame score returned to zero.

**Respotted black**
A black respotted to decide a frame when the scores are level after the final black has been resolved.

**Snookered**
A state in which the cue-ball does not have a direct path to both extreme edges of at least one ball on because another ball or balls obstruct the path.

**Striker**
The player whose turn it is to play.

**Visit / turn**
A period in which one player is at the table. A scoring visit may contain several consecutive strokes before control passes to the opponent.


## Rules and sources

The rule structure used in this study is based primarily on the official rules published by the **World Professional Billiards and Snooker Association (WPBSA)**.

### Primary rules

* [WPBSA — Official Rules of Snooker and English Billiards](https://wpbsa.com/wp-content/uploads/2198_WPBSA-Rulebook-2024-25.pdf)

Specific rules cited in the study include:

* Section 2, Rule 1 — **Frame**
* Section 3, Rule 1 — **Description**
* Section 3, Rule 3 — **Mode of Play**
* Section 3, Rule 4 — **End of Frame, Game or Match**
* Section 3, Rule 12 — **Snookered After a Foul**
* Section 3, Rule 13 — **Play Again**
* Section 3, Rule 14 — **Foul and a Miss**
* Section 3, Rule 17 — **Stalemate**
* Section 4, Rule 2 — **Conceding**

Where a rule has been used directly in the analysis, a link to the relevant page of the rulebook is included alongside it.

### Competitive examples

The study also uses a small number of real competitive examples where they illustrate a particular branch of the rules rather than merely providing trivia.

* [World Snooker Tour — *Warrior Wilson Battles Back To Make Final*](https://www.wst.tv/news/2025/march/21/warrior-wilson-battles-back-to-make-final/) — example of a foul ending a frame on a respotted black.
* [WPBSA — 147 Breaks](https://www.wpbsa.com/about-us/history/147-breaks/) — records of competitive breaks above 147, including Jamie Burnett's 148 and Ronnie O'Sullivan's 153.

### Scope

The Python representation in this notebook is deliberately simplified.

It is intended to expose the **structure of the game** rather than reproduce every provision of the rulebook or simulate the complete physics of a snooker table.

Where a rule cannot be determined from the simplified state — for example, whether the cue-ball is physically snookered after a foul — the model treats the relevant ruling as an external input rather than pretending it can infer information that has not been represented.
